# Phase 2 B1 — MetSeg Re-Extraction v2
## ROI Crop + Octant Spatial Pooling + Mask-weighted Pooling

**No training.** Inference only. Loads existing fine-tuned MetSeg checkpoints and re-extracts embeddings using three improvements over the v1 GAP approach:

| Fix | What it does | Impact |
|---|---|---|
| **ROI Crop** | Crops to WT bounding box + 8px padding, resizes to 64³ | Eliminates background brain signal |
| **Octant Pooling** | Divides spatial feature map into 8 sub-regions, pools each separately | Preserves spatial arrangement (shape, elongation) |
| **Mask-weighted Pooling** | Weights features by WT / TC / ET ground-truth masks | Separates biologically distinct subregion signals |

**Output:** `cnn_metseg_embeddings_fold{0,1,2}_v2.npz` per fold  
**Embedding dim:** `8 × C_oct  +  3 × C_neck` (revealed at runtime)

In [ ]:
!pip install -q monai
print('monai installed ✅')

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*Num foregrounds 0.*')
warnings.filterwarnings('ignore', message='.*non-tuple sequence.*')
warnings.filterwarnings('ignore', message='.*axcodes.*length.*')
warnings.filterwarnings('ignore', message='.*FutureWarning.*')

import os, json, shutil
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from collections import OrderedDict
from tqdm import tqdm

from monai.networks.nets import DynUNet
from monai.data import DataLoader, CacheDataset
import monai.transforms as T
from monai.transforms.compose import MapTransform
from monai.utils import ensure_tuple_rep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# DATA: cyprus-proteas-brain-mets dataset (same as all other notebooks)
# CKPTS: metseg-all-checkpoints dataset (upload metseg_fold{0,1,2}_best.pth)

DATA_ROOT = Path('/kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets')
CKPT_ROOT = Path('/kaggle/input/datasets/mohamedmohamed23/metseg-all-checkpoints')
OUTPUT_ROOT = Path('/kaggle/working/phase2_metseg_v2_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Model config (must match training) ─────────────────────────────────────
SEG_PARAMS = {
    'spatial_dims': 3, 'in_channels': 4, 'out_channels': 3,
    'kernel_size': [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],
    'strides':     [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
    'upsample_kernel_size': [[2,2,2],[2,2,2],[2,2,2],[2,2,2]],
    'deep_supervision': True, 'deep_supr_num': 3,
    'filters': [32, 64, 128, 256, 320], 'res_block': True, 'trans_bias': True,
}
ROI_SIZE    = (64, 64, 64)   # resize ROI crop to this for single forward pass
ROI_PADDING = 8              # voxels of padding around WT bounding box

print(f'DATA_ROOT : {DATA_ROOT} (exists={DATA_ROOT.exists()})')
print(f'CKPT_ROOT : {CKPT_ROOT} (exists={CKPT_ROOT.exists()})')
print('Checkpoints found:')
for f in sorted(CKPT_ROOT.rglob('*best*.pth')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')

In [ ]:
# ── Label converter: Cyprus {0,1,2,3} → BraTS-METS [WT, TC, ET] ────────────
class ConvertToMultiChannelBratsMetsd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img == 1) | (img == 3) | (img == 2),  # WT
                (img == 1) | (img == 3),                # TC
                img == 3,                                # ET
            ]
            d[key] = (torch.stack(result, dim=0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, axis=0).astype(np.float32))
        return d

# ── Path resolver ────────────────────────────────────────────────────────────
SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def resolve_path(root, rel):
    p = root / rel
    if p.exists(): return str(p)
    gz = str(p) + '.gz'
    if Path(gz).exists(): return gz
    nii_gz = str(p).replace('.nii.gz', '.nii_gz')
    if Path(nii_gz).exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(nii_gz, str(link))
        return str(link)
    parent = p.parent
    if parent.exists():
        target = p.name
        for f in parent.iterdir():
            if f.name.lower() == target.lower(): return str(f)
    raise FileNotFoundError(f'Not found: {rel}')

# ── Load splits ───────────────────────────────────────────────────────────────
splits_file = None
for name in ['data_splits.json', 'cv_splits_3fold.json', 'cv_splits.json']:
    candidate = DATA_ROOT / name
    if candidate.exists(): splits_file = candidate; break
if splits_file is None:
    for f in DATA_ROOT.rglob('*splits*.json'):
        splits_file = f; break

with open(splits_file) as f:
    raw_splits = json.load(f)
all_splits = raw_splits.get('3fold', raw_splits)
if 'fold_0' not in all_splits:
    all_splits = {k: v for k, v in raw_splits.items() if k.startswith('fold_')}
print(f'Splits: {splits_file.name}  |  folds: {list(all_splits.keys())}')

# ── Build scan dicts (all 170 scans) ─────────────────────────────────────────
def get_all_dicts():
    all_d, seen = [], set()
    for fk in all_splits:
        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:
            key = (scan['patient_dir'], scan['visit'])
            if key not in seen:
                seen.add(key)
                try:
                    all_d.append({
                        'image': [resolve_path(DATA_ROOT, scan['t1']),
                                  resolve_path(DATA_ROOT, scan['t1c']),
                                  resolve_path(DATA_ROOT, scan['t2']),
                                  resolve_path(DATA_ROOT, scan['fla'])],
                        'label': resolve_path(DATA_ROOT, scan['mask']),
                        'patient_dir': scan['patient_dir'],
                        'visit': scan['visit'],
                    })
                except FileNotFoundError:
                    pass
    return all_d

# ── Val transforms (same as training — no augmentation) ─────────────────────
val_transforms = T.Compose([
    T.LoadImaged(keys=['image', 'label']),
    T.EnsureChannelFirstd(keys=['image', 'label']),
    T.EnsureTyped(keys=['image', 'label']),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

all_dicts = get_all_dicts()
print(f'Total scans: {len(all_dicts)}')
print('Data + transforms ✅')

In [ ]:
# ── Model factory + checkpoint loader ─────────────────────────────────────────
def create_segmenter():
    return DynUNet(**SEG_PARAMS)

def load_fold_checkpoint(model, fold):
    """Load best checkpoint for the given fold."""
    candidates = list(CKPT_ROOT.rglob(f'*fold{fold}_best*.pth'))
    if not candidates:
        print(f'  ❌ Fold {fold}: checkpoint not found in {CKPT_ROOT}')
        return False
    ckpt_path = sorted(candidates)[0]
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    key = ('seg_state_dict' if 'seg_state_dict' in ckpt
           else 'model_state_dict' if 'model_state_dict' in ckpt
           else None)
    state = ckpt[key] if key else ckpt
    model.load_state_dict(state, strict=True)
    dice  = ckpt.get('best_dice', '?')
    epoch = ckpt.get('epoch', '?')
    print(f'  ✅ Fold {fold}: {ckpt_path.name}  (Dice={dice}, epoch={epoch})')
    return True

# ── Inspect model architecture to find octant + bottleneck hooks ───────────
seg_test = create_segmenter()
print(f'DynUNet: {sum(p.numel() for p in seg_test.parameters()):,} params')
print(f'  downsamples: {len(seg_test.downsamples)} blocks')
for i, block in enumerate(seg_test.downsamples):
    print(f'    [{i}]: {type(block).__name__}')
has_bottleneck = hasattr(seg_test, 'bottleneck')
print(f'  bottleneck attr: {has_bottleneck}')
del seg_test
print('Model architecture inspected ✅')

In [ ]:
# ── Probe actual channel counts at hook points for 64³ input ─────────────────
# This runs ONE forward pass with a dummy input to check spatial dims.

probe_model = create_segmenter().to(device).eval()
_probe = {}

def _hook_oct(m, i, o):
    feat = o[0] if isinstance(o, (list, tuple)) else o
    _probe['oct_shape'] = tuple(feat.shape)  # [B, C, H, W, D]

def _hook_neck(m, i, o):
    feat = o[0] if isinstance(o, (list, tuple)) else o
    _probe['neck_shape'] = tuple(feat.shape)

h1 = probe_model.downsamples[-1].register_forward_hook(_hook_oct)
h2 = (probe_model.bottleneck.register_forward_hook(_hook_neck)
      if hasattr(probe_model, 'bottleneck') else None)

with torch.no_grad():
    dummy = torch.zeros(1, 4, *ROI_SIZE, device=device)
    _ = probe_model(dummy)

h1.remove()
if h2: h2.remove()
del probe_model
torch.cuda.empty_cache()

# ── Compute embedding dims ──────────────────────────────────────────────────
oct_shape  = _probe['oct_shape']    # e.g. (1, 1024, 8, 8, 8)
neck_shape = _probe.get('neck_shape', oct_shape)  # fallback to oct layer

C_oct  = oct_shape[1]
C_neck = neck_shape[1]
OCTANT_DIM      = 8 * C_oct    # 8 octants × channels
MASK_WEIGHT_DIM = 3 * C_neck   # WT + TC + ET weighted pools
TOTAL_DIM       = OCTANT_DIM + MASK_WEIGHT_DIM

print(f'octant hook  → {oct_shape}   (spatial: {oct_shape[2]}×{oct_shape[3]}×{oct_shape[4]})')
print(f'neck   hook  → {neck_shape}  (spatial: {neck_shape[2]}×{neck_shape[3]}×{neck_shape[4]})')
print(f'Octant dim:       8 × {C_oct} = {OCTANT_DIM}')
print(f'Mask-weight dim:  3 × {C_neck} = {MASK_WEIGHT_DIM}')
print(f'Total emb dim:    {TOTAL_DIM}')

In [ ]:
# ── Core extraction function ─────────────────────────────────────────────────

def roi_crop_and_resize(image, label):
    """
    Crop image and label to the WT bounding box + ROI_PADDING, then
    resize both to ROI_SIZE (64³) for a single clean forward pass.
    Returns: (image_roi, label_roi) – both on the same device.
    """
    wt = label[0, 0]                          # [H, W, D] whole tumour mask
    nz = wt.nonzero(as_tuple=False)           # non-zero voxel coordinates

    if len(nz) == 0:
        # No tumour in this scan — use full image (rare edge case)
        img_crop = image
        lbl_crop = label
    else:
        lo = nz.min(0).values
        hi = nz.max(0).values
        sh = torch.tensor(wt.shape, device=wt.device)
        lo = torch.clamp(lo - ROI_PADDING, min=0)
        hi = torch.clamp(hi + ROI_PADDING + 1, max=sh)
        img_crop = image[:, :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
        lbl_crop = label[:,  :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]

    # Resize to fixed ROI_SIZE so one forward pass handles any tumour size
    img_roi = F.interpolate(img_crop, size=ROI_SIZE, mode='trilinear', align_corners=False)
    lbl_roi = F.interpolate(lbl_crop.float(), size=ROI_SIZE, mode='nearest')
    return img_roi, lbl_roi


def octant_pool(feat):
    """
    Divide spatial feature map into 8 (2×2×2) sub-regions, pool each.
    feat: [1, C, H, W, D]  →  returns 1-D tensor of size 8×C
    """
    H, W, D = feat.shape[2], feat.shape[3], feat.shape[4]
    pieces = []
    for hs in [slice(None, H // 2), slice(H // 2, None)]:
        for ws in [slice(None, W // 2), slice(W // 2, None)]:
            for ds in [slice(None, D // 2), slice(D // 2, None)]:
                region = feat[:, :, hs, ws, ds]
                pooled = F.adaptive_avg_pool3d(region, 1).flatten()  # C-dim
                pieces.append(pooled)
    return torch.cat(pieces)  # 8 × C


def mask_weighted_pool(feat, lbl_roi):
    """
    Weight feat by WT / TC / ET probability maps from the GT label.
    feat:    [1, C, H, W, D]
    lbl_roi: [1, 3, H_roi, W_roi, D_roi]  (already at ROI_SIZE)
    Returns: 1-D tensor of size 3×C
    """
    H, W, D = feat.shape[2], feat.shape[3], feat.shape[4]
    regions = []
    for ch in range(3):   # 0=WT, 1=TC, 2=ET
        prob = F.interpolate(lbl_roi[:, ch:ch+1], size=(H, W, D), mode='nearest')
        w_sum = (feat * prob).sum(dim=[0, 2, 3, 4])   # C-dim
        denom = prob.sum() + 1e-6
        regions.append(w_sum / denom)
    return torch.cat(regions)  # 3 × C


def extract_embeddings_v2(model, fold_label):
    """
    Full extraction pipeline:
      1. ROI crop → single 64³ forward pass
      2. Octant pooling on downsamples[-1]
      3. Mask-weighted pooling on bottleneck (or downsamples[-1] if no bottleneck)
      4. Concatenate → save .npz
    """
    model.eval(); model.to(device)
    _feats = {}   # storage updated by hooks

    def hook_oct(m, inp, out):
        feat = out[0] if isinstance(out, (list, tuple)) else out
        _feats['oct'] = feat.detach()

    def hook_neck(m, inp, out):
        feat = out[0] if isinstance(out, (list, tuple)) else out
        _feats['neck'] = feat.detach()

    h_oct  = model.downsamples[-1].register_forward_hook(hook_oct)
    h_neck = (model.bottleneck.register_forward_hook(hook_neck)
              if hasattr(model, 'bottleneck') else None)

    dataset = CacheDataset(all_dicts, val_transforms, cache_rate=0.3, num_workers=0)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    embeddings = {}
    skipped    = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc=f'Fold {fold_label}')):
            image   = batch['image'].to(device)   # [1, 4, H, W, D]
            label   = batch['label'].to(device)   # [1, 3, H, W, D]
            patient = batch['patient_dir'][0]
            visit   = batch['visit'][0]
            key     = f'{patient}__{visit}'

            try:
                # Step 1 — ROI crop + resize
                image_roi, label_roi = roi_crop_and_resize(image, label)

                # Step 2 — Single forward pass (hooks fire here)
                _feats.clear()
                _ = model(image_roi)

                oct_feat  = _feats.get('oct')
                neck_feat = _feats.get('neck', oct_feat)   # fallback if no bottleneck

                if oct_feat is None:
                    print(f'  ⚠️  No features captured for {key}, skipping')
                    skipped += 1; continue

                # Step 3 — Octant pooling (Fix A)
                oct_emb  = octant_pool(oct_feat)           # 8 × C_oct

                # Step 4 — Mask-weighted pooling (Fix B)
                mask_emb = mask_weighted_pool(neck_feat, label_roi)  # 3 × C_neck

                # Step 5 — Concatenate
                final_emb = torch.cat([oct_emb, mask_emb]).cpu().numpy()
                embeddings[key] = final_emb

                if i < 3:
                    print(f'    {key}: {final_emb.shape[0]}-dim '
                          f'(oct={oct_emb.shape[0]} + mask={mask_emb.shape[0]})')

            except Exception as e:
                print(f'  ⚠️  Error on {key}: {e}'); skipped += 1

    # Cleanup hooks
    h_oct.remove()
    if h_neck: h_neck.remove()

    print(f'  Extracted: {len(embeddings)} scans | Skipped: {skipped}')

    # Save .npz
    emb_dir = OUTPUT_ROOT / 'embeddings'
    emb_dir.mkdir(parents=True, exist_ok=True)
    out_path = emb_dir / f'cnn_metseg_embeddings_fold{fold_label}_v2.npz'
    np.savez(out_path, **embeddings)

    # Save metadata
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1],
                'embedding_dim': int(list(embeddings.values())[0].shape[0]),
                'oct_dim': OCTANT_DIM, 'mask_dim': MASK_WEIGHT_DIM}
            for k in embeddings}
    with open(emb_dir / f'cnn_metseg_embeddings_fold{fold_label}_v2_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    dim = list(embeddings.values())[0].shape[0]
    print(f'  ✅ Saved: {out_path.name}  ({len(embeddings)} × {dim}-dim)')
    return embeddings

print('Extraction function defined ✅')

In [ ]:
# ── MAIN — Extract all 3 folds ───────────────────────────────────────────────

print('=' * 60)
print('  MetSeg v2 Re-Extraction: ROI + Octant + Mask-weighted')
print('=' * 60)

all_results = {}

for fold in [0, 1, 2]:
    print(f'\n── Fold {fold} ─────────────────────────────────────')

    # Check if already done (e.g., re-run safety)
    out_path = OUTPUT_ROOT / 'embeddings' / f'cnn_metseg_embeddings_fold{fold}_v2.npz'
    if out_path.exists():
        data = np.load(out_path)
        print(f'  ⏭  Already exists: {len(data)} scans × {list(data.values())[0].shape[0]}-dim — skipping')
        all_results[fold] = dict(data)
        continue

    model = create_segmenter()
    ok = load_fold_checkpoint(model, fold)
    if not ok:
        print(f'  ⚠️  Fold {fold} skipped — checkpoint missing')
        continue

    model.to(device)
    embs = extract_embeddings_v2(model, fold)
    all_results[fold] = embs

    del model
    torch.cuda.empty_cache()

print('\n' + '=' * 60)
print('  All folds done!')
print('=' * 60)

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────

print('\n── Embedding Verification ──')
emb_dir = OUTPUT_ROOT / 'embeddings'

for fold in [0, 1, 2]:
    npz_path = emb_dir / f'cnn_metseg_embeddings_fold{fold}_v2.npz'
    if not npz_path.exists():
        print(f'  ❌ Fold {fold}: NOT FOUND'); continue

    data = np.load(npz_path)
    keys = list(data.keys())
    dim  = data[keys[0]].shape[0]
    vals = np.stack([data[k] for k in keys])
    norms = np.linalg.norm(vals, axis=1)
    print(f'  fold{fold}: {len(keys)} scans × {dim}-dim')
    print(f'         norm: [{norms.min():.2f}, {norms.max():.2f}]  std={norms.std():.3f}')
    print(f'         size: {npz_path.stat().st_size/1e6:.1f} MB')

print('\n── All output files ──')
total = 0
for f in sorted(OUTPUT_ROOT.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total += sz
        print(f'  {f.relative_to(OUTPUT_ROOT)}  ({sz/1e6:.1f} MB)')
print(f'Total: {total/1e6:.1f} MB')

print('\n── Download instructions ──')
print('Go to: Kaggle → Session files → phase2_metseg_v2_outputs/embeddings/')
print('Download:')
for fold in [0, 1, 2]:
    print(f'  cnn_metseg_embeddings_fold{fold}_v2.npz')
    print(f'  cnn_metseg_embeddings_fold{fold}_v2_meta.json')

In [ ]:
# ── Quick diversity check (are embeddings useful or collapsed?) ───────────────
import random

print('── Cosine Diversity Check ──')
print('(pair-wise cosine similarity on 50 random pairs per fold)')
print(f'{"Fold":<8} {"cos_mean":<12} {"diverse(<0.95) %"}')
print('-' * 38)

for fold in [0, 1, 2]:
    npz_path = emb_dir / f'cnn_metseg_embeddings_fold{fold}_v2.npz'
    if not npz_path.exists(): continue

    data  = np.load(npz_path)
    keys  = list(data.keys())
    vecs  = np.stack([data[k] for k in keys])
    # L2 normalise
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8
    vecs_n = vecs / norms

    n_pairs = min(50, len(keys) * (len(keys) - 1) // 2)
    sims = []
    pairs = random.sample([(i, j) for i in range(len(keys))
                            for j in range(i+1, len(keys))], n_pairs)
    for i, j in pairs:
        sims.append(float(np.dot(vecs_n[i], vecs_n[j])))

    cos_mean = np.mean(sims)
    diverse  = 100 * np.mean(np.array(sims) < 0.95)
    status   = '✅' if diverse > 20 else '⚠️ low diversity'
    print(f'  fold{fold}   {cos_mean:.3f}        {diverse:.1f}%   {status}')